# Cajas NN - CenterNet-lite para prompts MedSAM

Esta version reemplaza la red preliminar. La prueba anterior mostro algo valioso: la red si ve regiones vertebrales, pero los heatmaps por clase repetian centros. Por eso ahora usamos una formulacion mas parecida a la literatura de deteccion vertebral: detectar centros, estimar tamano/offset, predecir vertebras visibles y ordenar anatomica mente.

La salida sigue siendo una caja por vertebra, pensada como prompt previo a MedSAM. No se ejecuta MedSAM aqui.

<!-- codex-explicacion -->
Este notebook introduce la idea que cambio el proyecto: en vez de depender de reglas geometricas para ubicar vertebras, se entrena una red ligera tipo CenterNet para producir cajas automaticas.

Su papel es demostrar que una red pequena puede generar prompts utiles para MedSAM sin convertir este paso en un modelo pesado.


## 1. Configuracion

Se entrena con CUDA. El modelo es pequeno, pero usa una formulacion mas fuerte que la primera Tiny U-Net: focal loss para centros, regresion de tamano/offset y presencia visible. Si el entrenamiento deja de mejorar, se detiene por early stopping.

<!-- codex-explicacion -->
Se reunen rutas, semilla, dispositivo y parametros. La intencion fue que los cambios de experimento se hicieran desde un solo lugar.


In [ ]:
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REQUIERE_CUDA = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIERE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError("CUDA no esta disponible. Esta prueba esta pensada para GPU.")

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print("GPU:", torch.cuda.get_device_name(0))

DATASET_ROOT = Path("C:/Users/luisf/Downloads/ProyectoFinal/Scoliosis_Dataset")
MEDSAM_DATA_ROOT = Path("C:/Users/luisf/Downloads/ProyectoFinal/dataset_procesado_scoliosis_medsam/medsam")
LABELS_DICT_PATH = DATASET_ROOT / "diccionario_etiquetas_T1_T12_L1_L5.json"
RESULTADOS_DIR = Path("C:/Users/luisf/Downloads/ProyectoFinal/resultados_cajas_nn_centernet")
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 60
PACIENCIA = 10
LR = 8e-4
BASE_CH = 24

SIGMA_CENTRO = 4.5
BOX_EXPAND_W = 1.25
BOX_EXPAND_H = 1.20
PRESENCE_THR = 0.35
USAR_PRESENCE_EN_DECODIFICACION = False
MIN_VISIBLE_LABELS = 8
MAX_CANDIDATOS = 120
TOP_PICOS = 90
MIN_DIST_PICOS = 8
Y_MIN_ANATOMICO_MARGEN = 0.06
CRANEO_SCORE_FACTOR = 0.12
MAX_GAP_REL_DY = 2.40

# Ruta principal: 17 centros y etiquetado desde T1.
# La prueba con L5 como ancla queda solo como diagnostico porque desplazo casos buenos.
N_CAJAS_CAMINO = 17
ETIQUETADO_MODO = "top_anchor"  # top_anchor | bottom_anchor | auto_anchor
ESTRATEGIAS_ETIQUETADO = ["top_anchor"]
AUTO_BOTTOM_Y_REL = 0.82
AUTO_BOTTOM_MIN_EXTRA_BOXES = 4

LAMBDA_HEAT = 1.00
LAMBDA_WH = 0.40
LAMBDA_OFF = 0.12
LAMBDA_PRESENCE = 0.00

CKPT_PATH = RESULTADOS_DIR / "cajas_nn_centernet_lite_best.pt"

print("Device:", DEVICE)
print("IMG_SIZE:", IMG_SIZE)
print("Resultados:", RESULTADOS_DIR)


## 2. Carga de datos

Se reutiliza la exportacion MedSAM. `train` supervisa la red; `val` solo se usa para evaluar. Los IDs esperados son `T1..T12, L1..L5`.

<!-- codex-explicacion -->
Se leen los splits ya existentes. Este notebook no inventa una particion nueva: trabaja sobre el mismo train/val/test para mantener comparabilidad.

<!-- codex-normalizacion -->
Este notebook no vuelve a crear el dataset desde cero. Usa la exportaci?n MedSAM generada previamente en el notebook `03`: `dataset_procesado_scoliosis_medsam/medsam/train`, `val` y `test`.

Por eso, la normalizaci?n base ya viene heredada: im?genes a 1024x1024, formato compatible con MedSAM y m?scaras con IDs anat?micos preservados. Encima de eso, `CenterNet-lite` aplica una normalizaci?n interna adicional para su red, normalmente resize a `IMG_SIZE` y escalado por percentiles para estabilizar contraste.


In [ ]:
with open(LABELS_DICT_PATH, "r", encoding="utf-8") as f:
    LABELS_DICT = json.load(f)

MAPEO_ID = {int(k): v for k, v in LABELS_DICT["mascara_multiclase_id_png"].items()}
CLASES_OBJETIVO = [MAPEO_ID[i] for i in range(1, 18)]
N_CLASES = len(CLASES_OBJETIVO)
VERTEBRA_TO_ID = {nombre: idx for idx, nombre in MAPEO_ID.items() if idx != 0}
ID_TO_VERTEBRA = {idx: nombre for nombre, idx in VERTEBRA_TO_ID.items()}

SPLITS = ["train", "val", "test"]
EXTS_IMG = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]


def buscar_archivo_por_stem(carpeta, stem):
    for ext in EXTS_IMG:
        path = carpeta / f"{stem}{ext}"
        if path.exists():
            return path
    return None


def resolver_paths_split(split):
    split_dir = MEDSAM_DATA_ROOT / split
    prompts_path = split_dir / "prompts.json"
    image_dir = next((split_dir / d for d in ["images", "imgs", "image"] if (split_dir / d).is_dir()), None)
    mask_dir = next((split_dir / d for d in ["masks", "labels", "mask", "annotations"] if (split_dir / d).is_dir()), None)
    if image_dir is None or mask_dir is None or not prompts_path.exists():
        raise FileNotFoundError(f"Split incompleto: {split_dir}")
    return {"image_dir": image_dir, "mask_dir": mask_dir, "prompts_path": prompts_path}


def cargar_imagen(path):
    return np.array(Image.open(path).convert("RGB"))


def cargar_mascara(path):
    return np.array(Image.open(path)).astype(np.int32)


def normalizar_prompts(prompts_split):
    out = {}
    for item in prompts_split:
        patient_id = str(item.get("patient_id"))
        sub = {}
        prompts_item = item.get("prompts", {})
        iterable = prompts_item.values() if isinstance(prompts_item, dict) else prompts_item
        for info in iterable:
            if not isinstance(info, dict):
                continue
            vertebra = str(info.get("vertebra", "")).upper()
            if vertebra in CLASES_OBJETIVO and "bbox_xyxy" in info:
                sub[vertebra] = dict(info)
        if sub:
            out[patient_id] = sub
    return out


def resolver_paths_muestra(split, patient_id):
    info = SPLIT_INFO[split]
    path_img = buscar_archivo_por_stem(info["image_dir"], patient_id)
    path_mask = buscar_archivo_por_stem(info["mask_dir"], patient_id)
    if path_img is None or path_mask is None:
        raise FileNotFoundError(f"No encontre imagen/mascara para {patient_id}")
    return path_img, path_mask


def vertebras_presentes_en_mascara(mask):
    ids = sorted(int(v) for v in np.unique(mask) if 1 <= int(v) <= N_CLASES)
    return [ID_TO_VERTEBRA[i] for i in ids]


def filtrar_prompts_por_gt(prompts_sample, labels_gt):
    permitidas = set(labels_gt)
    return {v: info for v, info in prompts_sample.items() if v in permitidas}


def construir_gt_labels_dicc():
    dicc = {split: {} for split in SPLITS}
    filas = []
    for split in SPLITS:
        for patient_id in sorted(PROMPTS_DICC[split].keys()):
            _, path_mask = resolver_paths_muestra(split, patient_id)
            labels_gt = vertebras_presentes_en_mascara(cargar_mascara(path_mask))
            labels_prompt = sorted(PROMPTS_DICC[split][patient_id].keys(), key=lambda v: VERTEBRA_TO_ID[v])
            extras_prompt = [v for v in labels_prompt if v not in labels_gt]
            faltantes_prompt = [v for v in labels_gt if v not in labels_prompt]
            dicc[split][patient_id] = labels_gt
            filas.append({
                "split": split,
                "patient_id": patient_id,
                "n_gt": len(labels_gt),
                "n_prompts_json": len(labels_prompt),
                "n_prompts_extra_vs_gt": len(extras_prompt),
                "n_gt_sin_prompt_json": len(faltantes_prompt),
                "labels_gt": ",".join(labels_gt),
                "labels_prompt_extra_vs_gt": ",".join(extras_prompt),
                "labels_gt_sin_prompt_json": ",".join(faltantes_prompt),
            })
    return dicc, pd.DataFrame(filas)


SPLIT_INFO = {split: resolver_paths_split(split) for split in SPLITS}
PROMPTS = {}
for split in SPLITS:
    with open(SPLIT_INFO[split]["prompts_path"], "r", encoding="utf-8") as f:
        PROMPTS[split] = json.load(f)

PROMPTS_DICC = {split: normalizar_prompts(PROMPTS[split]) for split in SPLITS}
GT_LABELS_DICC, DF_AUDITORIA_GT = construir_gt_labels_dicc()

for split in SPLITS:
    print(f"{split}: {len(PROMPTS_DICC[split])} pacientes")
print("Clases:", CLASES_OBJETIVO)
display(
    DF_AUDITORIA_GT.groupby("split", as_index=False)
    .agg(
        pacientes=("patient_id", "count"),
        n_gt_promedio=("n_gt", "mean"),
        prompts_extra_vs_gt=("n_prompts_extra_vs_gt", "sum"),
        gt_sin_prompt_json=("n_gt_sin_prompt_json", "sum"),
    )
)


## 3. Plantilla de tamano

La red predice tamano, pero se conserva una plantilla mediana desde `train` como respaldo. Esto evita cajas absurdas si una prediccion de ancho/alto sale inestable.

<!-- codex-explicacion -->
La plantilla calcula tamanos tipicos de vertebra desde train. Sirve como respaldo anatomico cuando una prediccion individual sale inestable.

<!-- codex-normalizacion -->
La plantilla de tama?o solo tiene sentido porque las im?genes ya est?n en una escala com?n heredada de la exportaci?n MedSAM. Si cada radiograf?a estuviera en una resoluci?n distinta, esta plantilla mezclar?a anatom?a con cambios de escala.


In [ ]:
def construir_template_bbox_train(prompts_dicc, split_template="train"):
    filas = []
    for patient_id, prompts_sample in prompts_dicc[split_template].items():
        for vertebra, info in prompts_sample.items():
            x0, y0, x1, y1 = info["bbox_xyxy"]
            filas.append({
                "patient_id": patient_id,
                "vertebra": vertebra,
                "id_real": VERTEBRA_TO_ID[vertebra],
                "cx_rel": ((x0 + x1) / 2) / 1024,
                "cy_rel": ((y0 + y1) / 2) / 1024,
                "w_rel": (x1 - x0) / 1024,
                "h_rel": (y1 - y0) / 1024,
            })

    df = pd.DataFrame(filas)
    template = (
        df.groupby(["vertebra", "id_real"], as_index=False)
        .agg(cx_rel=("cx_rel", "median"), cy_rel=("cy_rel", "median"), w_rel=("w_rel", "median"), h_rel=("h_rel", "median"))
        .sort_values("id_real")
        .reset_index(drop=True)
    )
    return template, df


template_bbox, df_template = construir_template_bbox_train(PROMPTS_DICC)
display(template_bbox)


## 4. Targets CenterNet-lite

La red aprende cuatro cosas:

- `heatmap`: centro de cualquier vertebra visible.
- `wh`: ancho y alto de la caja en el centro.
- `offset`: correccion subpixel del centro.
- `presence`: que vertebras parecen visibles.

Durante entrenamiento se usan recortes verticales aleatorios para que aprenda imagenes parciales y no dependa de una posicion fija.

<!-- codex-explicacion -->
Se convierten cajas vertebrales en mapas de calor, tamanos y offsets. Esta formulacion evita predefinir anchors y permite detectar centros vertebrales.


In [ ]:
def transformar_prompts(prompts_sample, crop_y0=0, crop_y1=1024, flip=False):
    crop_h = max(crop_y1 - crop_y0, 1)
    scale_y = 1024 / crop_h
    out = {}

    for vertebra, info in prompts_sample.items():
        x0, y0, x1, y1 = [float(v) for v in info["bbox_xyxy"]]

        if flip:
            x0, x1 = 1023 - x1, 1023 - x0

        y0 = (y0 - crop_y0) * scale_y
        y1 = (y1 - crop_y0) * scale_y
        cy = (y0 + y1) / 2

        if cy < 0 or cy > 1023:
            continue

        x0 = float(np.clip(x0, 0, 1023))
        x1 = float(np.clip(x1, 0, 1023))
        y0 = float(np.clip(y0, 0, 1023))
        y1 = float(np.clip(y1, 0, 1023))

        if x1 - x0 < 4 or y1 - y0 < 4:
            continue

        nuevo = dict(info)
        nuevo["bbox_xyxy"] = [x0, y0, x1, y1]
        out[vertebra] = nuevo

    return out


def elegir_crop_vertical(prompts_sample, p_crop=0.40):
    if random.random() > p_crop:
        return 0, 1024

    centros = []
    for info in prompts_sample.values():
        x0, y0, x1, y1 = info["bbox_xyxy"]
        centros.append((y0 + y1) / 2)
    if len(centros) < 5:
        return 0, 1024

    for _ in range(20):
        crop_h = random.randint(680, 1024)
        y0 = random.randint(0, 1024 - crop_h)
        y1 = y0 + crop_h
        n_visible = sum(y0 <= c <= y1 for c in centros)
        if n_visible >= 5:
            return y0, y1

    return 0, 1024


def cargar_imagen_entrenamiento(split, patient_id, augment=False):
    path_img, _ = resolver_paths_muestra(split, patient_id)
    img = Image.open(path_img).convert("L")
    prompts = PROMPTS_DICC[split][patient_id]
    labels_gt = GT_LABELS_DICC.get(split, {}).get(patient_id, list(prompts.keys()))
    prompts_gt = filtrar_prompts_por_gt(prompts, labels_gt)
    if prompts_gt:
        prompts = prompts_gt

    crop_y0, crop_y1 = elegir_crop_vertical(prompts) if augment else (0, 1024)
    flip = bool(augment and random.random() < 0.50)

    if crop_y0 != 0 or crop_y1 != 1024:
        img = img.crop((0, crop_y0, 1024, crop_y1)).resize((1024, 1024), Image.BILINEAR)
    if flip:
        img = ImageOps.mirror(img)

    img_np = np.array(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32)
    if augment:
        alpha = np.random.uniform(0.88, 1.12)
        beta = np.random.uniform(-0.05, 0.05) * 255
        ruido = np.random.normal(0, np.random.uniform(0, 4), img_np.shape)
        img_np = np.clip(img_np * alpha + beta + ruido, 0, 255)

    p1, p99 = np.percentile(img_np, [1, 99.5])
    img_np = np.clip((img_np - p1) / (p99 - p1 + 1e-6), 0, 1)
    img_t = torch.from_numpy(img_np[None, ...].astype(np.float32))

    prompts_t = transformar_prompts(prompts, crop_y0=crop_y0, crop_y1=crop_y1, flip=flip)
    return img_t, prompts_t


def draw_gaussian(heat, cx, cy, sigma=SIGMA_CENTRO):
    radius = int(max(2, sigma * 3))
    x0 = max(0, int(cx) - radius)
    x1 = min(IMG_SIZE - 1, int(cx) + radius)
    y0 = max(0, int(cy) - radius)
    y1 = min(IMG_SIZE - 1, int(cy) + radius)
    if x1 <= x0 or y1 <= y0:
        return

    yy, xx = np.mgrid[y0:y1 + 1, x0:x1 + 1]
    g = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2)).astype(np.float32)
    heat[y0:y1 + 1, x0:x1 + 1] = np.maximum(heat[y0:y1 + 1, x0:x1 + 1], g)


def targets_desde_prompts(prompts_sample):
    heat = np.zeros((1, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    wh = np.zeros((2, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    off = np.zeros((2, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    mask = np.zeros((1, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    presence = np.zeros((N_CLASES,), dtype=np.float32)

    for vertebra, info in prompts_sample.items():
        if vertebra not in VERTEBRA_TO_ID:
            continue
        x0, y0, x1, y1 = [float(v) for v in info["bbox_xyxy"]]
        cx = ((x0 + x1) / 2) / 1024 * IMG_SIZE
        cy = ((y0 + y1) / 2) / 1024 * IMG_SIZE
        xi = int(np.clip(np.floor(cx), 0, IMG_SIZE - 1))
        yi = int(np.clip(np.floor(cy), 0, IMG_SIZE - 1))

        draw_gaussian(heat[0], cx, cy)
        heat[0, yi, xi] = 1.0
        mask[0, yi, xi] = 1.0
        wh[0, yi, xi] = np.clip((x1 - x0) / 1024, 0.01, 0.40)
        wh[1, yi, xi] = np.clip((y1 - y0) / 1024, 0.01, 0.40)
        off[0, yi, xi] = cx - xi
        off[1, yi, xi] = cy - yi
        presence[VERTEBRA_TO_ID[vertebra] - 1] = 1.0

    return {
        "heat": torch.from_numpy(heat),
        "wh": torch.from_numpy(wh),
        "off": torch.from_numpy(off),
        "mask": torch.from_numpy(mask),
        "presence": torch.from_numpy(presence),
    }


class VertebraCenterDataset(Dataset):
    def __init__(self, split, augment=False, patient_ids=None):
        self.split = split
        self.augment = augment
        self.patient_ids = list(patient_ids) if patient_ids is not None else sorted(PROMPTS_DICC[split].keys())

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        patient_id = self.patient_ids[idx]
        img_t, prompts_t = cargar_imagen_entrenamiento(self.split, patient_id, augment=self.augment)
        target = targets_desde_prompts(prompts_t)
        return img_t, target, {"split": self.split, "patient_id": patient_id}


ds_train = VertebraCenterDataset("train", augment=True)
ds_val = VertebraCenterDataset("val", augment=False)
print("Train:", len(ds_train), "Val:", len(ds_val))


## 5. Modelo

El backbone es una U-Net pequena. Las cabezas imitan CenterNet: centro, tamano y offset. Una cabeza global predice presencia de cada nivel vertebral para manejar imagenes parciales.

<!-- codex-explicacion -->
La arquitectura es ligera a proposito: debe actuar como paso previo a MedSAM, no reemplazarlo ni volver el pipeline demasiado costoso.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class CenterNetLite(nn.Module):
    def __init__(self, base=BASE_CH):
        super().__init__()
        self.e1 = ConvBlock(1, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.e4 = ConvBlock(base * 4, base * 6)
        self.b = ConvBlock(base * 6, base * 8)

        self.u4 = ConvBlock(base * 8 + base * 6, base * 6)
        self.u3 = ConvBlock(base * 6 + base * 4, base * 4)
        self.u2 = ConvBlock(base * 4 + base * 2, base * 2)
        self.u1 = ConvBlock(base * 2 + base, base)

        self.heat_head = nn.Conv2d(base, 1, 1)
        self.wh_head = nn.Conv2d(base, 2, 1)
        self.off_head = nn.Conv2d(base, 2, 1)
        self.presence_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(base * 8, N_CLASES),
        )

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        e4 = self.e4(F.max_pool2d(e3, 2))
        b = self.b(F.max_pool2d(e4, 2))

        u4 = F.interpolate(b, size=e4.shape[-2:], mode="bilinear", align_corners=False)
        u4 = self.u4(torch.cat([u4, e4], dim=1))
        u3 = F.interpolate(u4, size=e3.shape[-2:], mode="bilinear", align_corners=False)
        u3 = self.u3(torch.cat([u3, e3], dim=1))
        u2 = F.interpolate(u3, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        u2 = self.u2(torch.cat([u2, e2], dim=1))
        u1 = F.interpolate(u2, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        u1 = self.u1(torch.cat([u1, e1], dim=1))

        return {
            "heat": self.heat_head(u1),
            "wh": self.wh_head(u1),
            "off": self.off_head(u1),
            "presence": self.presence_head(b),
        }


model = CenterNetLite().to(DEVICE)
print("Parametros:", round(sum(p.numel() for p in model.parameters()) / 1e6, 3), "M")


## 6. Perdidas y entrenamiento

La perdida principal es focal loss para centros, que castiga falsos positivos como craneo/cuello. `wh` y `offset` solo se evalúan en pixeles centro.

Antes de construir los targets, cada imagen filtra sus prompts contra las etiquetas presentes en su GT. Si una imagen como `S_80` solo tiene `T1..T8` etiquetadas, el entrenamiento no inventa perdida para `T9..L5` en esa muestra.

La cabeza `presence` queda temporalmente sin peso de perdida (`LAMBDA_PRESENCE = 0`) y no decide el tramo visible. En la corrida parcial vimos que no estaba aprendiendo; usarla para cortar `T1..L5` introducia errores. Primero queremos que el centro y la secuencia anatomica sean solidos.

<!-- codex-explicacion -->
La perdida combina deteccion de centros y regresion de caja. El objetivo no es segmentar aun, sino producir prompts suficientemente buenos.


In [ ]:
def focal_loss_centernet(logits, target):
    pred = torch.sigmoid(logits).clamp(1e-4, 1 - 1e-4)
    pos = (target >= 0.999).float()
    neg = (target < 0.999).float()
    neg_weights = torch.pow(1 - target, 4)

    pos_loss = -torch.log(pred) * torch.pow(1 - pred, 2) * pos
    neg_loss = -torch.log(1 - pred) * torch.pow(pred, 2) * neg_weights * neg
    num_pos = pos.sum().clamp(min=1.0)
    return (pos_loss.sum() + neg_loss.sum()) / num_pos


def regression_loss_at_centers(pred_logits, target, mask, kind="sigmoid_l1"):
    if kind == "sigmoid_l1":
        pred = torch.sigmoid(pred_logits)
    else:
        pred = pred_logits
    mask2 = mask.expand_as(target)
    denom = mask2.sum().clamp(min=1.0)
    return (torch.abs(pred - target) * mask2).sum() / denom


def loss_batch(outputs, target):
    heat_loss = focal_loss_centernet(outputs["heat"], target["heat"])
    wh_loss = regression_loss_at_centers(outputs["wh"], target["wh"], target["mask"], kind="sigmoid_l1")
    off_loss = regression_loss_at_centers(outputs["off"], target["off"], target["mask"], kind="sigmoid_l1")
    presence_loss = F.binary_cross_entropy_with_logits(outputs["presence"], target["presence"])
    total = (
        LAMBDA_HEAT * heat_loss +
        LAMBDA_WH * wh_loss +
        LAMBDA_OFF * off_loss +
        LAMBDA_PRESENCE * presence_loss
    )
    return total, {
        "heat": float(heat_loss.detach().cpu()),
        "wh": float(wh_loss.detach().cpu()),
        "off": float(off_loss.detach().cpu()),
        "presence": float(presence_loss.detach().cpu()),
    }


def mover_target_device(target):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in target.items()}


def crear_loaders():
    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE.type == "cuda"))
    val_loader = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader


@torch.no_grad()
def evaluar_loss_loader(loader):
    model.eval()
    losses = []
    partes = []
    for x, target, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        target = mover_target_device(target)
        outputs = model(x)
        loss, parts = loss_batch(outputs, target)
        losses.append(float(loss.detach().cpu()))
        partes.append(parts)
    if not losses:
        return np.nan, {}
    mean_parts = {k: float(np.mean([p[k] for p in partes])) for k in partes[0].keys()}
    return float(np.mean(losses)), mean_parts


def entrenar_modelo():
    train_loader, val_loader = crear_loaders()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    best_val = np.inf
    sin_mejora = 0
    hist = []
    t0 = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for x, target, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            x = x.to(DEVICE, non_blocking=True)
            target = mover_target_device(target)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(x)
                loss, parts = loss_batch(outputs, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.detach().cpu()))

        val_loss, val_parts = evaluar_loss_loader(val_loader)
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "val_loss": val_loss,
            "val_heat": val_parts.get("heat", np.nan),
            "val_wh": val_parts.get("wh", np.nan),
            "val_off": val_parts.get("off", np.nan),
            "val_presence": val_parts.get("presence", np.nan),
            "tiempo_min": (time.perf_counter() - t0) / 60,
        }
        hist.append(row)
        print(f"epoch={epoch:03d} train={row['train_loss']:.5f} val={val_loss:.5f} heat={row['val_heat']:.4f} wh={row['val_wh']:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            sin_mejora = 0
            torch.save({"model": model.state_dict(), "config": {"IMG_SIZE": IMG_SIZE, "N_CLASES": N_CLASES}}, CKPT_PATH)
        else:
            sin_mejora += 1
            if sin_mejora >= PACIENCIA:
                print("Early stopping.")
                break

    hist = pd.DataFrame(hist)
    hist.to_csv(RESULTADOS_DIR / "historial_entrenamiento_centernet.csv", index=False)
    return hist


hist_entrenamiento = entrenar_modelo()
display(hist_entrenamiento.tail())

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
print("Mejor checkpoint cargado:", CKPT_PATH)


## 7. Decodificacion anatomica

El heatmap entrega candidatos de centros. Luego se selecciona una secuencia monotona usando programacion dinamica.

La prueba con `bottom_anchor` mostro algo util: en algunos casos hay cajas buenas pero el nombre anatomico queda desplazado. Sin embargo, usar `L5` como ancla por defecto rompio casos buenos como `N_12` y `S_187`.

Por eso la ruta principal vuelve a ser conservadora:

- Detectar 17 centros.
- Etiquetar desde arriba: `T1..L5`.
- Usar la metrica flexible solo como diagnostico de desfase, no como reemplazo de la metrica estricta.

<!-- codex-explicacion -->
Las predicciones deben ordenarse y asignarse a etiquetas vertebrales. Aqui empieza a verse que la localizacion y el nombre anatomico son problemas relacionados pero distintos.


In [ ]:
def imagen_inferencia_tensor(img_rgb):
    gray = np.array(Image.fromarray(img_rgb).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32)
    p1, p99 = np.percentile(gray, [1, 99.5])
    gray = np.clip((gray - p1) / (p99 - p1 + 1e-6), 0, 1)
    return torch.from_numpy(gray[None, None, ...].astype(np.float32)).to(DEVICE)


@torch.no_grad()
def predecir_outputs(img_rgb):
    model.eval()
    x = imagen_inferencia_tensor(img_rgb)
    out = model(x)
    return {
        "heat": torch.sigmoid(out["heat"])[0, 0].detach().cpu().numpy(),
        "wh": torch.sigmoid(out["wh"])[0].detach().cpu().numpy(),
        "off": torch.sigmoid(out["off"])[0].detach().cpu().numpy(),
        "presence": torch.sigmoid(out["presence"])[0].detach().cpu().numpy(),
    }


def extraer_picos(score_map, n_picos=TOP_PICOS, min_dist=MIN_DIST_PICOS, thr_rel=0.12):
    work = score_map.astype(np.float32).copy()
    picos = []
    max0 = float(work.max())
    if max0 <= 0:
        return picos
    thr = max(max0 * thr_rel, float(np.percentile(work, 90)))

    for _ in range(n_picos):
        idx = int(np.argmax(work))
        y, x = np.unravel_index(idx, work.shape)
        score = float(work[y, x])
        if score < thr:
            break
        picos.append({"x": int(x), "y": int(y), "score": score})
        y0 = max(0, y - min_dist)
        y1 = min(work.shape[0], y + min_dist + 1)
        x0 = max(0, x - min_dist)
        x1 = min(work.shape[1], x + min_dist + 1)
        work[y0:y1, x0:x1] = -np.inf
    return picos


def estimar_y_min_anatomico(img_rgb):
    """Estima un limite superior para penalizar craneo/cuello sin usar GT."""
    gray = np.array(Image.fromarray(img_rgb).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32)
    valores = gray[gray > 0]
    if len(valores) == 0:
        return 0.0, {"motivo": "sin_valores"}

    thr = max(5.0, float(np.percentile(valores, 8)))
    active = (gray > thr).astype(np.float32)
    width = active.mean(axis=1)
    kernel = np.ones(21, dtype=np.float32) / 21
    width_s = np.convolve(width, kernel, mode="same")

    h = IMG_SIZE
    top_med = float(np.median(width_s[: int(0.16 * h)]))
    search0 = int(0.12 * h)
    search1 = int(0.55 * h)
    search = width_s[search0:search1]
    if len(search) == 0:
        return 0.0, {"motivo": "sin_search"}

    mid_p85 = float(np.percentile(search, 85))
    umbral_ensanche = max(0.24, top_med * 1.35)
    if top_med > 0.30 or mid_p85 < umbral_ensanche:
        return 0.0, {"motivo": "sin_craneo_claro", "top_med": top_med, "mid_p85": mid_p85}

    idx = np.where(search > umbral_ensanche)[0]
    if len(idx) == 0:
        return 0.0, {"motivo": "sin_ensanche", "top_med": top_med, "mid_p85": mid_p85}

    y_ensanche = float(search0 + idx[0])
    y_min = max(0.0, y_ensanche - Y_MIN_ANATOMICO_MARGEN * h)
    return y_min, {"motivo": "craneo_probable", "top_med": top_med, "mid_p85": mid_p85, "y_ensanche": y_ensanche}


def seleccionar_camino_dp(candidatos, n_pasos=N_CAJAS_CAMINO, max_gap_rel=MAX_GAP_REL_DY):
    if len(candidatos) == 0:
        raise RuntimeError("No hay candidatos de centro.")

    candidatos = sorted(candidatos, key=lambda c: (c["y"], c["x"]))[:MAX_CANDIDATOS]
    n = len(candidatos)
    k = min(int(n_pasos), n)
    if k <= 0:
        raise RuntimeError("No hay suficientes candidatos.")

    xs = np.array([c["x"] for c in candidatos], dtype=np.float32)
    ys = np.array([c["y"] for c in candidatos], dtype=np.float32)
    sc = np.array([c["score"] for c in candidatos], dtype=np.float32)

    template = template_bbox.sort_values("id_real").reset_index(drop=True)
    cy_template = template["cy_rel"].to_numpy(dtype=np.float32) * IMG_SIZE
    dy_template = np.diff(cy_template)
    dy_default = float(np.median(dy_template))

    dp = np.full((k, n), -1e9, dtype=np.float32)
    prev = np.full((k, n), -1, dtype=np.int32)
    dp[0] = sc

    for j in range(1, k):
        expected_dy = float(dy_template[j - 1]) if j - 1 < len(dy_template) else dy_default
        expected_dy = max(expected_dy, 4.0)
        min_gap = max(3.0, expected_dy * 0.35)
        max_gap = max(min_gap + 1.0, expected_dy * max_gap_rel)

        for i in range(n):
            dy = ys[i] - ys[:i]
            valid = (dy >= min_gap) & (dy <= max_gap)
            if not valid.any():
                continue
            dx = np.abs(xs[i] - xs[:i])
            dy_pen = np.abs(dy - expected_dy) / expected_dy
            dx_pen = dx / max(IMG_SIZE * 0.22, 1.0)
            trans = dp[j - 1, :i] - 0.34 * dy_pen - 0.08 * dx_pen
            trans[~valid] = -1e9
            best = int(np.argmax(trans))
            dp[j, i] = sc[i] + trans[best]
            prev[j, i] = best

    end = int(np.argmax(dp[k - 1]))
    if dp[k - 1, end] < -1e8 and max_gap_rel < 8.0:
        return seleccionar_camino_dp(candidatos, n_pasos=n_pasos, max_gap_rel=8.0)

    path = [end]
    for j in range(k - 1, 0, -1):
        end = int(prev[j, end])
        if end < 0:
            break
        path.append(end)
    path = path[::-1]

    if len(path) != k:
        orden = np.argsort(sc)[-k:]
        path = sorted(orden.tolist(), key=lambda i: ys[i])

    return [candidatos[i] for i in path]


def bbox_clip(bbox, H, W):
    x0, y0, x1, y1 = [int(round(float(v))) for v in bbox]
    x0, x1 = np.clip([x0, x1], 0, W - 1)
    y0, y1 = np.clip([y0, y1], 0, H - 1)
    if x1 <= x0:
        x1 = min(W - 1, x0 + 1)
    if y1 <= y0:
        y1 = min(H - 1, y0 + 1)
    return [int(x0), int(y0), int(x1), int(y1)]


def decidir_modo_etiquetado_auto(camino):
    if not camino:
        return "top_anchor", {"motivo": "sin_camino"}
    y_bottom_rel = max(float(c["y"]) for c in camino) / IMG_SIZE
    extra_boxes = max(0, len(camino) - N_CLASES)
    if y_bottom_rel >= AUTO_BOTTOM_Y_REL and extra_boxes >= AUTO_BOTTOM_MIN_EXTRA_BOXES:
        return "bottom_anchor", {"motivo": "region_inferior_visible", "y_bottom_rel": y_bottom_rel, "extra_boxes": extra_boxes}
    return "top_anchor", {"motivo": "sin_ancla_inferior_fuerte", "y_bottom_rel": y_bottom_rel, "extra_boxes": extra_boxes}


def elegir_segmento_y_labels(camino, modo_etiquetado="auto_anchor"):
    modo = modo_etiquetado or ETIQUETADO_MODO
    if modo not in ESTRATEGIAS_ETIQUETADO:
        raise ValueError(f"Estrategia de etiquetado no soportada: {modo}")

    camino = sorted(camino, key=lambda c: (c["y"], c["x"]))
    modo_resuelto = modo
    info = {"modo_solicitado": modo}
    if modo == "auto_anchor":
        modo_resuelto, info_auto = decidir_modo_etiquetado_auto(camino)
        info.update(info_auto)

    n_use = min(N_CLASES, len(camino))
    if modo_resuelto == "bottom_anchor":
        segmento = camino[-n_use:]
        labels = list(range(N_CLASES - n_use, N_CLASES))
    else:
        segmento = camino[:n_use]
        labels = list(range(n_use))

    info["modo_resuelto"] = modo_resuelto
    info["n_camino"] = len(camino)
    info["n_prompts"] = len(segmento)
    return segmento, labels, info


def construir_prompts_desde_segmento(segmento, labels, img_rgb, presence, modo_info):
    H, W = img_rgb.shape[:2]
    template = template_bbox.sort_values("id_real").reset_index(drop=True)
    prompts = {}
    filas = []

    for orden, (cand, label_idx) in enumerate(zip(segmento, labels)):
        vertebra = ID_TO_VERTEBRA[label_idx + 1]
        trow = template[template["id_real"] == label_idx + 1].iloc[0]
        cx = cand["x"] / IMG_SIZE * W
        cy = cand["y"] / IMG_SIZE * H
        pred_w = np.clip(cand["wh_rel"][0], 0.03, 0.28) * W
        pred_h = np.clip(cand["wh_rel"][1], 0.025, 0.18) * H
        tpl_w = float(trow["w_rel"] * W)
        tpl_h = float(trow["h_rel"] * H)
        bw = (0.65 * pred_w + 0.35 * tpl_w) * BOX_EXPAND_W
        bh = (0.65 * pred_h + 0.35 * tpl_h) * BOX_EXPAND_H
        bbox = bbox_clip([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], H, W)

        prompts[vertebra] = {
            "vertebra": vertebra,
            "id_real": label_idx + 1,
            "bbox_xyxy": bbox,
            "confianza_nn": float(cand["score"]),
            "confianza_original": float(cand.get("score_original", cand["score"])),
            "presencia_nn": float(presence[label_idx]) if label_idx < len(presence) else np.nan,
            "estrategia_etiquetado": modo_info["modo_solicitado"],
            "estrategia_resuelta": modo_info["modo_resuelto"],
            "orden_caja": int(orden),
            "n_camino": int(modo_info["n_camino"]),
            "y_min_anatomico": float(cand.get("y_min_anatomico", 0.0)),
            "prompt_origen": "cajas_nn_centernet_lite",
        }
        filas.append({
            "vertebra": vertebra,
            "id_real": label_idx + 1,
            "cx": cx,
            "cy": cy,
            "confianza": float(cand["score"]),
            "confianza_original": float(cand.get("score_original", cand["score"])),
            "presencia": float(presence[label_idx]) if label_idx < len(presence) else np.nan,
            "penalizado_craneo": bool(cand.get("penalizado_craneo", False)),
            "y_min_anatomico": float(cand.get("y_min_anatomico", 0.0)),
            "bbox_xyxy": bbox,
            "estrategia_etiquetado": modo_info["modo_solicitado"],
            "estrategia_resuelta": modo_info["modo_resuelto"],
            "orden_caja": int(orden),
            "n_camino": int(modo_info["n_camino"]),
        })

    return prompts, pd.DataFrame(filas).sort_values("id_real").reset_index(drop=True)


def prompts_desde_outputs(img_rgb, out, modo_etiquetado=None):
    heat, wh_map, off_map, presence = out["heat"], out["wh"], out["off"], out["presence"]
    picos = extraer_picos(heat)
    y_min_hm, info_anatomico = estimar_y_min_anatomico(img_rgb)

    candidatos = []
    for p in picos:
        x, y = p["x"], p["y"]
        cx_hm = x + float(off_map[0, y, x])
        cy_hm = y + float(off_map[1, y, x])
        score = float(p["score"])

        if cy_hm < y_min_hm:
            if cy_hm < y_min_hm - IMG_SIZE * 0.06:
                continue
            score *= CRANEO_SCORE_FACTOR

        candidatos.append({
            "x": cx_hm,
            "y": cy_hm,
            "score": score,
            "score_original": float(p["score"]),
            "penalizado_craneo": bool(cy_hm < y_min_hm),
            "y_min_anatomico": float(y_min_hm),
            "wh_rel": [float(wh_map[0, y, x]), float(wh_map[1, y, x])],
        })

    camino = seleccionar_camino_dp(candidatos, n_pasos=N_CAJAS_CAMINO)
    segmento, labels, modo_info = elegir_segmento_y_labels(camino, modo_etiquetado=modo_etiquetado)
    prompts, df_centros = construir_prompts_desde_segmento(segmento, labels, img_rgb, presence, modo_info)

    df_candidatos = pd.DataFrame(candidatos)
    df_candidatos.attrs["info_anatomico"] = info_anatomico
    df_candidatos.attrs["modo_info"] = modo_info
    return prompts, df_centros, df_candidatos


def generar_prompts_nn(split, patient_id, modo_etiquetado=None):
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    img = cargar_imagen(path_img)
    mask = cargar_mascara(path_mask)
    out = predecir_outputs(img)
    prompts, df_centros, df_candidatos = prompts_desde_outputs(img, out, modo_etiquetado=modo_etiquetado)
    return img, mask, prompts, df_centros, df_candidatos, out


## 8. Metricas por etiqueta GT

La evaluacion principal sigue siendo estricta por etiqueta: `T1` se compara contra `T1`, `T2` contra `T2`, etc. Solo entran al promedio las etiquetas presentes en la mascara GT de cada imagen.

Ademas se calcula una metrica flexible por etiqueta GT: para cada vertebra anotada se busca cual de todas las cajas generadas cae mejor sobre esa mascara. Esta metrica no reemplaza la estricta; sirve para saber si el problema es la caja o el nombre anatomico asignado por el decoder.

Ejemplo: si `GT T6` tiene IoU bajo con la caja llamada `T6`, pero alto con la caja llamada `T9`, entonces la red vio una vertebra util, pero el orden/nombre quedo desplazado.

<!-- codex-explicacion -->
La evaluacion por etiqueta evita esconder errores en el promedio global. Es clave para saber si el modelo falla mas en cervical/toracica/lumbar o en casos parciales.


In [ ]:
def mask_binaria_vertebra(mask, vertebra):
    return (mask == VERTEBRA_TO_ID[vertebra]).astype(np.uint8)


def vertebras_gt_eval(mascara_gt):
    return vertebras_presentes_en_mascara(mascara_gt)


def metricas_bbox_contra_gt(gt, bbox):
    x0, y0, x1, y1 = [int(v) for v in bbox]
    cover = np.zeros_like(gt, dtype=bool)
    cover[y0:y1 + 1, x0:x1 + 1] = True
    inter = int(np.logical_and(gt, cover).sum())
    union = int(np.logical_or(gt, cover).sum())
    bbox_area = int(cover.sum())
    total_gt = int(gt.sum())

    ys, xs = np.where(gt)
    if len(xs) == 0:
        center_error = np.nan
    else:
        cx_gt, cy_gt = xs.mean(), ys.mean()
        cx_box, cy_box = (x0 + x1) / 2, (y0 + y1) / 2
        center_error = math.sqrt((cx_box - cx_gt) ** 2 + (cy_box - cy_gt) ** 2)

    return {
        "bbox_iou": inter / union if union else 0.0,
        "bbox_recall": inter / total_gt if total_gt else 0.0,
        "bbox_precision": inter / bbox_area if bbox_area else 0.0,
        "center_error_px": center_error,
    }


def mejor_prompt_para_gt(prompts_auto, gt):
    mejor = None
    mejor_key = None
    for vertebra_prompt, info in prompts_auto.items():
        if "bbox_xyxy" not in info:
            continue
        met = metricas_bbox_contra_gt(gt, info["bbox_xyxy"])
        center = met["center_error_px"]
        center_key = -center if np.isfinite(center) else -1e9
        key = (met["bbox_iou"], met["bbox_recall"], center_key)
        if mejor is None or key > mejor_key:
            mejor = {
                "prompt_flexible_vertebra": vertebra_prompt,
                "prompt_flexible_id": VERTEBRA_TO_ID.get(vertebra_prompt, np.nan),
                "bbox_iou_flexible": met["bbox_iou"],
                "bbox_recall_flexible": met["bbox_recall"],
                "bbox_precision_flexible": met["bbox_precision"],
                "center_error_flexible_px": met["center_error_px"],
                "confianza_flexible": info.get("confianza_nn", np.nan),
            }
            mejor_key = key
    return mejor


def evaluar_cajas(prompts_auto, mascara_gt, vertebras_eval=None):
    if vertebras_eval is None:
        vertebras_eval = vertebras_gt_eval(mascara_gt)

    filas = []
    for vertebra in vertebras_eval:
        gt = mask_binaria_vertebra(mascara_gt, vertebra).astype(bool)
        total_gt = int(gt.sum())
        if total_gt == 0:
            continue

        mejor = mejor_prompt_para_gt(prompts_auto, gt)
        if vertebra in prompts_auto:
            met_exacta = metricas_bbox_contra_gt(gt, prompts_auto[vertebra]["bbox_xyxy"])
            estado_prompt = "evaluado_gt"
            confianza_nn = prompts_auto[vertebra].get("confianza_nn", np.nan)
            presencia_nn = prompts_auto[vertebra].get("presencia_nn", np.nan)
        else:
            met_exacta = {
                "bbox_iou": 0.0,
                "bbox_recall": 0.0,
                "bbox_precision": 0.0,
                "center_error_px": np.nan,
            }
            estado_prompt = "gt_sin_prompt"
            confianza_nn = np.nan
            presencia_nn = np.nan

        fila = {
            "vertebra": vertebra,
            "id_real": VERTEBRA_TO_ID[vertebra],
            "bbox_iou": met_exacta["bbox_iou"],
            "bbox_recall": met_exacta["bbox_recall"],
            "bbox_precision": met_exacta["bbox_precision"],
            "center_error_px": met_exacta["center_error_px"],
            "confianza_nn": confianza_nn,
            "presencia_nn": presencia_nn,
            "estado_prompt": estado_prompt,
        }

        if mejor is not None:
            fila.update(mejor)
            fila["flexible_misma_etiqueta"] = bool(mejor["prompt_flexible_vertebra"] == vertebra)
            fila["desfase_id_flexible"] = int(mejor["prompt_flexible_id"] - VERTEBRA_TO_ID[vertebra])
            fila["mejora_iou_flexible"] = float(mejor["bbox_iou_flexible"] - met_exacta["bbox_iou"])
        else:
            fila.update({
                "prompt_flexible_vertebra": "",
                "prompt_flexible_id": np.nan,
                "bbox_iou_flexible": np.nan,
                "bbox_recall_flexible": np.nan,
                "bbox_precision_flexible": np.nan,
                "center_error_flexible_px": np.nan,
                "confianza_flexible": np.nan,
                "flexible_misma_etiqueta": False,
                "desfase_id_flexible": np.nan,
                "mejora_iou_flexible": np.nan,
            })

        filas.append(fila)

    if not filas:
        return pd.DataFrame(columns=[
            "vertebra", "id_real", "bbox_iou", "bbox_recall", "bbox_precision",
            "center_error_px", "confianza_nn", "presencia_nn", "estado_prompt",
            "prompt_flexible_vertebra", "prompt_flexible_id", "bbox_iou_flexible",
            "bbox_recall_flexible", "bbox_precision_flexible", "center_error_flexible_px",
            "flexible_misma_etiqueta", "desfase_id_flexible", "mejora_iou_flexible",
        ])

    return pd.DataFrame(filas).sort_values("id_real").reset_index(drop=True)


def resumen_etiquetas_prompts(prompts_auto, vertebras_gt):
    labels_prompts = sorted([v for v in prompts_auto.keys() if v in VERTEBRA_TO_ID], key=lambda v: VERTEBRA_TO_ID[v])
    labels_gt = list(vertebras_gt)
    labels_eval = [v for v in labels_prompts if v in labels_gt]
    labels_extra = [v for v in labels_prompts if v not in labels_gt]
    labels_faltantes = [v for v in labels_gt if v not in labels_prompts]
    return labels_prompts, labels_eval, labels_extra, labels_faltantes


def describir_desfases_flexibles(df, min_mejora=0.05):
    if df.empty or "prompt_flexible_vertebra" not in df:
        return ""
    filas = []
    for _, row in df.iterrows():
        prompt = row.get("prompt_flexible_vertebra", "")
        if not prompt or prompt == row["vertebra"]:
            continue
        mejora = float(row.get("mejora_iou_flexible", 0.0))
        iou_flex = float(row.get("bbox_iou_flexible", 0.0))
        if mejora >= min_mejora:
            filas.append(f"{row['vertebra']}<-{prompt}({iou_flex:.2f})")
    return ";".join(filas)


def evaluar_muestra_nn(split, patient_id, modo_etiquetado=None):
    t0 = time.perf_counter()
    img, mask, prompts, df_centros, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=modo_etiquetado)
    vertebras_gt = vertebras_gt_eval(mask)
    df = evaluar_cajas(prompts, mask, vertebras_eval=vertebras_gt)
    labels_prompts, labels_eval, labels_extra, labels_faltantes = resumen_etiquetas_prompts(prompts, vertebras_gt)
    tipo_real = "escoliosis" if str(patient_id).startswith("S_") else "normal"

    return {
        "split": split,
        "patient_id": patient_id,
        "tipo_real": tipo_real,
        "modo_evaluacion": "estricta_y_flexible_solo_etiquetas_gt",
        "estrategia_etiquetado": df_centros["estrategia_etiquetado"].iloc[0] if not df_centros.empty else (modo_etiquetado or ETIQUETADO_MODO),
        "estrategia_resuelta": df_centros["estrategia_resuelta"].iloc[0] if not df_centros.empty else "",
        "n_camino": int(df_centros["n_camino"].iloc[0]) if not df_centros.empty and "n_camino" in df_centros else 0,
        "n_gt": int(len(vertebras_gt)),
        "n_prompts": int(len(prompts)),
        "n_prompts_evaluados": int(len(labels_eval)),
        "n_prompts_extra_no_evaluados": int(len(labels_extra)),
        "n_gt_sin_prompt": int(len(labels_faltantes)),
        "labels_gt": ",".join(vertebras_gt),
        "labels_prompts_evaluados": ",".join(labels_eval),
        "labels_prompts_extra_no_evaluados": ",".join(labels_extra),
        "labels_gt_sin_prompt": ",".join(labels_faltantes),
        "tiempo_s": time.perf_counter() - t0,
        "bbox_iou_promedio": float(df["bbox_iou"].mean()) if not df.empty else np.nan,
        "bbox_recall_promedio": float(df["bbox_recall"].mean()) if not df.empty else np.nan,
        "bbox_precision_promedio": float(df["bbox_precision"].mean()) if not df.empty else np.nan,
        "n_vertebras_iou_mayor_02": int((df["bbox_iou"] > 0.20).sum()) if not df.empty else 0,
        "center_error_px": float(df["center_error_px"].mean()) if not df.empty else np.nan,
        "bbox_iou_flexible_promedio": float(df["bbox_iou_flexible"].mean()) if not df.empty else np.nan,
        "bbox_recall_flexible_promedio": float(df["bbox_recall_flexible"].mean()) if not df.empty else np.nan,
        "bbox_precision_flexible_promedio": float(df["bbox_precision_flexible"].mean()) if not df.empty else np.nan,
        "n_vertebras_flexible_iou_mayor_02": int((df["bbox_iou_flexible"] > 0.20).sum()) if not df.empty else 0,
        "center_error_flexible_px": float(df["center_error_flexible_px"].mean()) if not df.empty else np.nan,
        "n_flexible_misma_etiqueta": int(df["flexible_misma_etiqueta"].sum()) if not df.empty else 0,
        "mejora_iou_flexible_promedio": float(df["mejora_iou_flexible"].mean()) if not df.empty else np.nan,
        "desfase_id_flexible_promedio": float(df["desfase_id_flexible"].mean()) if not df.empty else np.nan,
        "desfases_flexibles": describir_desfases_flexibles(df),
        "confianza_media": float(df["confianza_nn"].mean()) if "confianza_nn" in df else np.nan,
    }, df


def evaluar_split_nn(split="val", patient_ids=None, modo_etiquetado=None):
    if patient_ids is None:
        patient_ids = sorted(PROMPTS_DICC[split].keys())

    resumen, detalles, errores = [], [], []
    for pid in tqdm(patient_ids, desc=f"Evaluando {split}"):
        try:
            row, df = evaluar_muestra_nn(split, pid, modo_etiquetado=modo_etiquetado)
            resumen.append(row)
            detalles.append(df.assign(split=split, patient_id=pid))
        except Exception as exc:
            errores.append({"split": split, "patient_id": pid, "error": repr(exc)})

    df_resumen = pd.DataFrame(resumen)
    df_detalle = pd.concat(detalles, ignore_index=True) if detalles else pd.DataFrame()
    df_errores = pd.DataFrame(errores)

    df_resumen.to_csv(RESULTADOS_DIR / f"cajas_nn_centernet_val_resumen.csv", index=False)
    df_detalle.to_csv(RESULTADOS_DIR / f"cajas_nn_centernet_val_detalle.csv", index=False)
    df_errores.to_csv(RESULTADOS_DIR / f"cajas_nn_centernet_val_errores.csv", index=False)
    return df_resumen, df_detalle, df_errores



def comparar_estrategias_etiquetado(split="val", patient_ids=None, estrategias=None):
    """Diagnostico opcional. La ruta principal usa solo top_anchor."""
    estrategias = estrategias or ESTRATEGIAS_ETIQUETADO
    frames = []
    for estrategia in estrategias:
        df_res, _, _ = evaluar_split_nn(split, patient_ids=patient_ids, modo_etiquetado=estrategia)
        if not df_res.empty:
            frames.append(df_res)

    df_cmp = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    out_path = RESULTADOS_DIR / f"cajas_nn_centernet_{split}_comparativo_estrategias.csv"
    df_cmp.to_csv(out_path, index=False)
    print("Comparativo guardado:", out_path)
    return df_cmp


## 9. Prueba de viabilidad

Se prueban los mismos casos criticos. Si la metrica y las imagenes mejoran, se puede pasar a todo `val`.

<!-- codex-explicacion -->
Esta prueba contesta si vale la pena seguir con una red de cajas. Si aqui no hubiera senal, no tendria sentido conectarla con MedSAM.


In [ ]:
PACIENTES_PRUEBA = ["N_12", "N_32", "S_187", "S_130", "S_80", "S_190"]
patient_ids_eval = [p for p in PACIENTES_PRUEBA if p in PROMPTS_DICC["val"]]

df_nn_resumen, df_nn_detalle, df_nn_errores = evaluar_split_nn("val", patient_ids_eval, modo_etiquetado=ETIQUETADO_MODO)
display(df_nn_resumen)

if not df_nn_resumen.empty:
    display(
        df_nn_resumen.groupby(["tipo_real", "estrategia_resuelta"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            tiempo_s=("tiempo_s", "mean"),
            bbox_iou_promedio=("bbox_iou_promedio", "mean"),
            bbox_iou_flexible_promedio=("bbox_iou_flexible_promedio", "mean"),
            bbox_recall_promedio=("bbox_recall_promedio", "mean"),
            bbox_recall_flexible_promedio=("bbox_recall_flexible_promedio", "mean"),
            n_vertebras_iou_mayor_02=("n_vertebras_iou_mayor_02", "mean"),
            n_vertebras_flexible_iou_mayor_02=("n_vertebras_flexible_iou_mayor_02", "mean"),
            center_error_px=("center_error_px", "mean"),
            center_error_flexible_px=("center_error_flexible_px", "mean"),
        )
    )

# Diagnostico opcional, no se ejecuta por defecto porque bottom_anchor rompio casos buenos.
# df_cmp_estrategias = comparar_estrategias_etiquetado("val", patient_ids_eval, estrategias=["top_anchor", "bottom_anchor"])

display(df_nn_errores)


## 10. Visualizacion unica

La vista queda en dos paneles:

- Izquierda: metrica estricta, cada etiqueta GT contra el prompt del mismo nombre.
- Derecha: metrica flexible, cada etiqueta GT contra el mejor prompt generado.

Si `IoU_flex` sube mucho frente a `IoU_GT`, el problema principal es el etiquetado/anclaje anatomico, no la deteccion de cajas.

<!-- codex-explicacion -->
La visualizacion cualitativa se usa para detectar fallas que las metricas no explican bien, como cajas desplazadas hacia abajo o repetidas.


In [ ]:
def visualizar_nn(split, patient_id, guardar=True, modo_etiquetado=None):
    img, mask, prompts, df_centros, df_candidatos, out = generar_prompts_nn(split, patient_id, modo_etiquetado=modo_etiquetado)
    vertebras_gt = vertebras_gt_eval(mask)
    df_eval = evaluar_cajas(prompts, mask, vertebras_eval=vertebras_gt)
    labels_prompts, labels_eval, labels_extra, labels_faltantes = resumen_etiquetas_prompts(prompts, vertebras_gt)
    tab = df_eval.set_index("vertebra") if not df_eval.empty else pd.DataFrame()

    iou_gt = float(df_eval["bbox_iou"].mean()) if not df_eval.empty else np.nan
    iou_fx = float(df_eval["bbox_iou_flexible"].mean()) if not df_eval.empty else np.nan
    recall_gt = float(df_eval["bbox_recall"].mean()) if not df_eval.empty else np.nan
    recall_fx = float(df_eval["bbox_recall_flexible"].mean()) if not df_eval.empty else np.nan
    estrategia = df_centros["estrategia_etiquetado"].iloc[0] if not df_centros.empty else (modo_etiquetado or ETIQUETADO_MODO)
    estrategia_resuelta = df_centros["estrategia_resuelta"].iloc[0] if not df_centros.empty else ""

    fig, ax = plt.subplots(1, 2, figsize=(16, 9))
    overlay = np.zeros_like(img)
    overlay[..., 0] = (mask > 0).astype(np.uint8) * 255

    ax[0].imshow(img)
    ax[0].imshow(overlay, alpha=0.18)
    ax[0].set_title("Estricta: GT vs misma etiqueta")
    ax[0].axis("off")
    for vertebra, info in prompts.items():
        if vertebra not in vertebras_gt:
            continue
        x0, y0, x1, y1 = info["bbox_xyxy"]
        ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="lime", linewidth=1.2))
        iou = float(tab.loc[vertebra, "bbox_iou"]) if vertebra in tab.index else 0.0
        ax[0].text(x0, max(0, y0 - 3), f"{vertebra} {iou:.2f}", fontsize=7, color="white", bbox=dict(facecolor="black", alpha=0.45, pad=1))

    ax[1].imshow(img)
    ax[1].imshow(overlay, alpha=0.18)
    ax[1].set_title("Flexible: GT vs mejor prompt")
    ax[1].axis("off")
    for _, row in df_eval.iterrows():
        gt_v = row["vertebra"]
        best_v = row.get("prompt_flexible_vertebra", "")
        if not best_v or best_v not in prompts:
            continue
        x0, y0, x1, y1 = prompts[best_v]["bbox_xyxy"]
        color = "gold" if best_v != gt_v else "lime"
        ax[1].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=1.2))
        ax[1].text(
            x0,
            max(0, y0 - 3),
            f"GT {gt_v}<-{best_v} {float(row['bbox_iou_flexible']):.2f}",
            fontsize=7,
            color="white",
            bbox=dict(facecolor="black", alpha=0.45, pad=1),
        )

    y_min_txt = "" if df_centros.empty else f" | y_min={df_centros['y_min_anatomico'].iloc[0]:.0f}"
    fig.suptitle(
        f"{patient_id} | {estrategia}->{estrategia_resuelta} | IoU_GT={iou_gt:.3f} | IoU_flex={iou_fx:.3f} "
        f"| recall_GT={recall_gt:.3f} | recall_flex={recall_fx:.3f} | GT={len(vertebras_gt)} | extra={len(labels_extra)}" + y_min_txt,
        fontsize=13,
    )
    plt.tight_layout()

    if guardar:
        out_path = RESULTADOS_DIR / f"visual_centernet_val_{patient_id}_{estrategia}.png"
        plt.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.show()

    display(df_centros)
    display(pd.DataFrame([{
        "patient_id": patient_id,
        "estrategia": estrategia,
        "estrategia_resuelta": estrategia_resuelta,
        "labels_gt": ",".join(vertebras_gt),
        "labels_prompts_evaluados": ",".join(labels_eval),
        "labels_prompts_extra_no_evaluados": ",".join(labels_extra),
        "labels_gt_sin_prompt": ",".join(labels_faltantes),
        "desfases_flexibles": describir_desfases_flexibles(df_eval),
    }]))
    display(df_eval)
    return prompts, df_eval, df_centros, df_candidatos


for pid in patient_ids_eval:
    visualizar_nn("val", pid, modo_etiquetado=ETIQUETADO_MODO)


## 11. Exportar prompts

Cuando las cajas se vean viables, esta funcion exporta prompts para MedSAM. No ejecuta MedSAM.

<!-- codex-explicacion -->
Los prompts exportados permiten que MedSAM use las cajas sin volver a ejecutar todo el entrenamiento del detector.


In [ ]:
def exportar_prompts_nn(split="val", patient_ids=None, modo_etiquetado=None):
    modo = modo_etiquetado or ETIQUETADO_MODO
    if patient_ids is None:
        patient_ids = sorted(PROMPTS_DICC[split].keys())

    salida = []
    for pid in tqdm(patient_ids, desc=f"Exportando prompts NN {split} {modo}"):
        img, mask, prompts, df_centros, _, _ = generar_prompts_nn(split, pid, modo_etiquetado=modo)
        salida.append({
            "patient_id": pid,
            "estrategia_etiquetado": modo,
            "estrategia_resuelta": df_centros["estrategia_resuelta"].iloc[0] if not df_centros.empty else "",
            "prompts": prompts,
        })

    out_path = RESULTADOS_DIR / f"prompts_nn_centernet_{split}_{modo}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(salida, f, ensure_ascii=False, indent=2)
    print("Guardado:", out_path)
    return out_path


# exportar_prompts_nn("val", modo_etiquetado=ETIQUETADO_MODO)


## 12. Criterio de decision

La ruta principal queda conservadora:

- `top_anchor` es el valor por defecto porque conserva los casos buenos.
- `bottom_anchor` queda como diagnostico, no como estrategia final.
- Si `IoU_flex` es mucho mayor que `IoU_GT`, la red detecto cajas utiles pero el nombre anatomico quedo desplazado.
- El siguiente salto debe ser elegir mejor el tramo/ancla, no volver a forzar `L5` globalmente.

Para MedSAM, las cajas utiles importan mucho; para reporte por vertebra, el nombre anatomico debe quedar bien anclado.

<!-- codex-explicacion -->
Esta seccion resume si el detector es suficientemente bueno para pasar al siguiente notebook. La decision no depende solo de una metrica, tambien de estabilidad visual.


<!-- codex-cierre-etapa -->
## Cierre de etapa y siguiente paso

Esta etapa prob? que una red ligera tipo CenterNet pod?a detectar centros y cajas vertebrales de forma autom?tica. Fue un cambio importante frente a las reglas geom?tricas, porque el modelo empez? a aprender patrones visuales de las v?rtebras en lugar de depender solo de heur?sticas.

La limitaci?n fue que detectar cajas no era todav?a el resultado final: hab?a que comprobar si esas cajas realmente funcionaban como prompts para MedSAM y si mejoraban la segmentaci?n vertebral completa. Por eso el siguiente paso fue integrar detector de cajas + MedSAM en un pipeline entrenado y evaluado de principio a fin.

**Siguiente notebook:** `05_nn_sam_entrenamiento_completo.ipynb`.
